# Word2Vec: Skip-gram with Negative Sampling (SGNS)

## Part 1: Step-by-Step Explanation

### Step 1: The Skip-gram Concept

The Skip-gram model flips the traditional language modeling approach. Instead of using context words to predict a missing center word (which is what the Continuous Bag of Words, or CBOW, model does), Skip-gram **uses a single center word to predict its surrounding context words.**

If our sentence is: *"The quick brown fox jumps over the lazy dog."*
If the target word is **"fox"**, the model tries to predict the surrounding context words: ["The", "quick", "brown", "jumps", "over"].

### Step 2: Creating the Training Data (Sliding Window)

To train the model, we slide a "window" (e.g., size $L=2$) across our text corpus.
For the phrase *"quick brown fox jumps over"*:

* Center word: **fox**

* Context words (Window size 2): **quick, brown, jumps, over**

We create positive training pairs: `(fox, quick)`, `(fox, brown)`, `(fox, jumps)`, `(fox, over)`.
For each pair, the network's goal is to output a high probability.


### Step 3: The Bottleneck (Why we need Negative Sampling)

A standard neural network would use a **Softmax** layer at the end to output a probability distribution over the *entire vocabulary* (which might be 100,000+ words).

To update the weights for just *one* training pair `(fox, jumps)`, the Softmax function forces us to calculate the dot product of "fox" against all 100,000 words in the vocabulary, sum them up for the denominator, and update every single weight. This is computationally impossible to do efficiently on massive datasets.

### Step 4: The Genius of Negative Sampling

Negative Sampling solves this by reframing the problem. Instead of a massive multi-class classification problem ("Which of these 100,000 words is the context word?"), it turns it into a lightweight **Binary Classification** problem ("Is this specific pair a real context pair, Yes or No?").

1. **Positive Sample:** We take our real pair `(fox, jumps)`. The model should output `1` (True).

2. **Negative Samples:** We randomly pick $K$ "fake" words from our vocabulary that are *not* in the context window (e.g., $K=5$). Let's say we pick "apple", "car", "cloud", "matrix", "shoe".

3. We create fake pairs: `(fox, apple)`, `(fox, car)`, etc. The model should output `0` (False) for these.

Now, instead of updating 100,000 weights, we only update the weights for the $1$ positive word and the $K$ negative words. We reduced the computational cost by 99.99%.


## Part 2: The Mathematics of SGNS

### 1. The Embedding Matrices

Word2Vec maintains *two* distinct embedding matrices:

* $V$: The "Target" or "Center" word matrix. (Size: `Vocabulary Size x Embedding Dimension`).

* $U$: The "Context" word matrix. (Size: `Vocabulary Size x Embedding Dimension`).

Let $v_c$ be the vector for the center word (from matrix $V$).
Let $u_o$ be the vector for the actual context word (from matrix $U$).
Let $u_k$ be the vector for a randomly drawn negative word (from matrix $U$).

### 2. The Binary Prediction (Sigmoid)

To predict the probability that a pair is "real" ($y=1$), we take the dot product of their vectors and pass it through a Sigmoid function $\sigma(x) = \frac{1}{1 + e^{-x}}$.

* Probability that the real pair is valid: $P(y=1 | c, o) = \sigma(v_c \cdot u_o)$

* Probability that a noise pair is invalid: $P(y=0 | c, k) = 1 - \sigma(v_c \cdot u_k) = \sigma(-v_c \cdot u_k)$

*(Note: A mathematical property of sigmoid is that* $1 - \sigma(x) = \sigma(-x)$*)*.


### 3. The Objective Function (Loss)

We want to **maximize** the probability of the real context word, and **maximize** the probability that the fake words are classified as fake.

Because we assume these are independent events, we multiply the probabilities together. To make the math easier for derivatives, we take the **Logarithm** of this product, giving us the Log-Likelihood objective function ($J$):

$$
J = \log \sigma(v_c \cdot u_o) + \sum_{k=1}^{K} \mathbb{E}_{w_k \sim P_n(w)} \left[ \log \sigma(-v_c \cdot u_k) \right]
$$

**Breaking down the formula:**

* $\log \sigma(v_c \cdot u_o)$: This forces the dot product of the center word and the *true* context word to be as large and positive as possible (pushing their vectors closer together in space).

* $\sum_{k=1}^{K}$: We sum over our $K$ negative samples.

* $w_k \sim P_n(w)$: We draw negative words based on a noise distribution $P_n(w)$ (usually the unigram frequency of the word raised to the 3/4 power, which prevents ultra-rare words from never being sampled, and ultra-common words like "the" from being sampled too often).

* $\log \sigma(-v_c \cdot u_k)$: This forces the dot product of the center word and the *fake* word to be as large and negative as possible (pushing their vectors far apart).


### 4. Minimizing the Loss (Gradient Descent)

In standard PyTorch/TensorFlow implementations, optimization algorithms minimize loss. Therefore, we simply take the negative of $J$ to get the **Negative Log-Likelihood (NLL) Loss**:

$$
\mathcal{L}_{SGNS} = - \log \sigma(v_c \cdot u_o) - \sum_{k=1}^{K} \log \sigma(-v_c \cdot u_k)
$$

During backpropagation, the model calculates the partial derivatives of this loss with respect to $v_c$, $u_o$, and the $u_k$ vectors, updating only this tiny subset of weights.

By iteratively pulling context words closer to the center word and pushing random words away, the geometric space organizes itself perfectly so that words with similar contexts end up clustered together, effectively capturing semantic meaning.
